In [1]:
import pickle
import mlflow
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

with open('../src/features/X_y.pkl', 'rb') as f:
    X, y = pickle.load(f)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Fake in train: {y_train.sum()}, Fake in test: {y_test.sum()}")

c:\Users\MH Mazin\fake-job-detector\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: (14304, 504), Test: (3576, 504)
Fake in train: 693, Fake in test: 173


In [2]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("fake_job_detection")

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred)
    }

2026/07/08 23:32:16 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/08 23:32:16 INFO mlflow.store.db.utils: Updating database tables
2026/07/08 23:32:18 INFO mlflow.tracking.fluent: Experiment with name 'fake_job_detection' does not exist. Creating a new experiment.


In [3]:
with mlflow.start_run(run_name="logistic_regression"):
    params = {"C": 1.0, "max_iter": 1000, "class_weight": "balanced"}
    mlflow.log_params(params)
    
    lr = LogisticRegression(**params)
    lr.fit(X_train, y_train)
    
    metrics = evaluate_model(lr, X_test, y_test)
    mlflow.log_metrics(metrics)
    
    print(f"LR - Accuracy: {metrics['accuracy']:.3f}, F1: {metrics['f1']:.3f}")

LR - Accuracy: 0.868, F1: 0.377


In [4]:
with mlflow.start_run(run_name="random_forest"):
    params = {"n_estimators": 100, "max_depth": 10, "class_weight": "balanced", "random_state": 42}
    mlflow.log_params(params)
    
    rf = RandomForestClassifier(**params)
    rf.fit(X_train, y_train)
    
    metrics = evaluate_model(rf, X_test, y_test)
    mlflow.log_metrics(metrics)
    
    print(f"RF - Accuracy: {metrics['accuracy']:.3f}, F1: {metrics['f1']:.3f}")

RF - Accuracy: 0.857, F1: 0.314


In [6]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
experiment = client.get_experiment_by_name("fake_job_detection")
runs = client.search_runs(experiment_ids=[experiment.experiment_id])

print("All runs:")
for run in runs:
    f1 = run.data.metrics.get("f1", 0)
    print(f"  {run.info.run_name}: F1={f1:.3f}")

best = max(runs, key=lambda r: r.data.metrics.get("f1", 0))
print(f"\nBest: {best.info.run_name}, F1={best.data.metrics['f1']:.3f}")

All runs:
  random_forest: F1=0.314
  logistic_regression: F1=0.377

Best: logistic_regression, F1=0.377


In [7]:
import os
os.makedirs('../src/models', exist_ok=True)

with open('../src/models/model.pkl', 'wb') as f:
    pickle.dump(rf, f)
with open('../src/models/vectorizer.pkl', 'wb') as f:
    with open('../src/features/tfidf_vectorizer.pkl', 'rb') as v:
        pickle.dump(pickle.load(v), f)

print("Model saved")

Model saved
